# Pipeline step 2: mechanistic diagnosis — what's different under the hood for `culture=us`?

**Runs on a Colab T4 GPU** (`colab exec -s <session> -f <this file>`), loading the real
`CohereLabs/tiny-aya-water` HF checkpoint (not the GGUF quantization used elsewhere in
this project) so we can read internal activations.

**Known limitation, stated up front:** full `CircuitKIT` attention-head/MLP circuit
*discovery* requires `transformer_lens.HookedTransformer`, and Tiny Aya's architecture
is not in TransformerLens's supported model registry (checked directly — no `aya` or
`cohere` match among its 249 official models). So this notebook does **not** run
`circuitkit.discover()` on the real model. Instead it uses `safetune`'s
model-agnostic, hook-based primitives (`extract_refusal_direction`,
`identify_safety_neurons`), which work on any `nn.Module` — a real circuit-level
(direction + neuron) analysis of the actual Tiny Aya checkpoint, just at a coarser
grain (linear direction + MLP neurons) than full attention-head circuit discovery.
`pipeline2b_circuitkit_demo.ipynb` separately runs true `CircuitKIT` discovery on a
TransformerLens-supported stand-in model, to show that part of the methodology works,
clearly labeled as not being about Tiny Aya.

**What this notebook actually tests:**
1. Extract the "refusal direction" (the residual-stream direction separating harmful
   from harmless-twin prompts) separately in English and in Japanese, from
   `culture=neutral` prompts only (the cleanest condition).
2. Rank the MLP neurons most aligned with that direction (`identify_safety_neurons`) —
   candidate targets for a circuit-restricted fix in pipeline step 4.
3. **The actual diagnostic:** project `culture=us` harmful prompts' activations onto
   that same refusal direction, and compare to `culture=neutral` harmful prompts. If
   `us`-framing systematically pushes activation *away* from the refusal direction,
   that's a mechanistic account of why it jailbreaks more (pipeline 1 found this is
   real and language-independent, OR=1.98, p<0.0001 pooled across `en`/`ja`).
4. **Cross-language transfer check:** apply the *English*-derived direction to
   *Japanese* prompts' activations (and vice versa). If the `us`-vs-`neutral` gap still
   shows up, the mechanism is language-invariant — a single real target to fix, not two
   separate language-specific quirks.


## 0. Setup

In [ ]:
import json
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetune", "transformers", "accelerate", "huggingface_hub"], check=True)

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
import safetune.steer as steer
from safetune.instrumentation.interpret import identify_safety_neurons, SafetyNeuronConfig
import scipy.stats as stats

# Tiny Aya is a gated HF checkpoint; HF_TOKEN is passed in via `colab exec --env`.
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("WARNING: no HF_TOKEN env var found; gated model load will likely fail.")

MODEL_ID = "CohereLabs/tiny-aya-water"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
if device != "cuda":
    print("WARNING: no GPU detected — this will be slow and/or may not reflect the intended T4 run.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.

device: cuda

## 1. Load the real Tiny Aya HF checkpoint (bfloat16)

In [ ]:

print("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map=device)
model.eval()
print("Model loaded:", MODEL_ID)


Loading tokenizer/model...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded:

CohereLabs/tiny-aya-water

## 2. Load the real prompt sample

40 prompts per (language x culture x kind) cell, pulled directly from this project's
`observations.csv` (`framing=control`, to isolate the culture effect from framing),
uploaded alongside this notebook as `pipeline2_prompts.json`.

In [ ]:
with open("/content/pipeline2_prompts.json", encoding="utf-8") as f:
    data = json.load(f)
for lang in ["en", "ja"]:
    print(lang, {k: v for k, v in data[lang].items() if k.startswith("n_")})


en

{'n_harmful_neutral': 8, 'n_twin_neutral': 8, 'n_harmful_us': 8}

ja

{'n_harmful_neutral': 8, 'n_twin_neutral': 8, 'n_harmful_us': 8}

## 3. Helper: last-token hidden states at every layer

Standard convention for direction/activation analysis: the last real (non-padding)
token's residual-stream value at each layer.

In [ ]:

def last_token_hidden_states(prompts, batch_size=8):
    all_layers = None
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        messages = [[{"role": "user", "content": p}] for p in batch]
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
        enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
        last_idx = enc["attention_mask"].sum(dim=1) - 1
        if all_layers is None:
            all_layers = [[] for _ in out.hidden_states]
        for layer_i, hs in enumerate(out.hidden_states):
            row_idx = torch.arange(hs.size(0))
            all_layers[layer_i].append(hs[row_idx, last_idx].float().cpu())
    return [torch.cat(chunks, dim=0) for chunks in all_layers]


## 4. Extract the refusal direction + rank safety neurons, per language

Contrast: `kind=harmful` vs `kind=twin`, both `culture=neutral` — the cleanest
harmful-vs-harmless pair, uncontaminated by the culture framing under study.

In [ ]:
results = {}
directions_per_lang = {}
# select_directions=False: skip the expensive per-layer validation sweep (which ablates
# every candidate layer and scores KL-divergence via generation — this OOM'd on the T4
# at the default settings) and use the documented middle-layer fallback instead. This
# is a real, supported config path (safetune.steer.RefusalDirectionConfig docstring),
# not a workaround that changes what's being measured in principle.
direction_config = steer.RefusalDirectionConfig(select_directions=False)

for lang in ["en", "ja"]:
    print(f"\n=== {lang}: extracting refusal direction (harmful vs twin, culture=neutral) ===")
    harmful = data[lang]["harmful_neutral"]
    twin = data[lang]["twin_neutral"]
    direction, layer_idx, all_layer_directions = steer.extract_refusal_direction(
        model, tokenizer, harmful_prompts=harmful, harmless_prompts=twin, config=direction_config,
    )
    print(f"  selected layer_idx={layer_idx}")
    directions_per_lang[lang] = (direction, layer_idx, all_layer_directions)
    torch.cuda.empty_cache()

    print("  ranking safety neurons (method=weight, from all-layer directions)...")
    report = identify_safety_neurons(model, all_layer_directions, SafetyNeuronConfig(method="weight", top_k_per_layer=16))
    top_layers = sorted(report.per_layer.items(), key=lambda kv: -max(abs(s) for _, s in kv[1]) if kv[1] else 0)[:5]
    print(f"  top 5 layers by max |score|: {[layer for layer, _ in top_layers]}")
    results[lang] = {
        "layer_idx": layer_idx,
        "top_layers_by_safety_score": [int(layer) for layer, _ in top_layers],
        "safety_neurons_top_layer": [[int(i), float(s)] for i, s in report.per_layer.get(top_layers[0][0], [])[:16]] if top_layers else [],
    }
results



=== en: extracting refusal direction (harmful vs twin, culture=neutral) ===

Calibrate [harmless]:   0%|          | 0/1 [00:00<?, ?batch/s]

Calibrate [harmful]:   0%|          | 0/1 [00:00<?, ?batch/s]

  selected layer_idx=18

  ranking safety neurons (method=weight, from all-layer directions)...

  top 5 layers by max |score|: [23, 24, 27, 15, 18]

... (3 outputs left out)

  selected layer_idx=18

  ranking safety neurons (method=weight, from all-layer directions)...

  top 5 layers by max |score|: [18, 25, 28, 7, 15]

{'en': {'layer_idx': 18,
  'top_layers_by_safety_score': [23, 24, 27, 15, 18],
  'safety_neurons_top_layer': [[4198, 0.32846102118492126],
   [10045, 0.21222341060638428],
   [2297, 0.18264585733413696],
   [2517, 0.16521579027175903],
   [8225, 0.15693753957748413],
   [35, 0.15670610964298248],
   [3741, 0.15396757423877716],
   [265, 0.15223166346549988],
   [7830, 0.14674922823905945],
   [653, 0.14152975380420685],
   [7842, 0.1413089483976364],
   [1869, 0.14031895995140076],
   [8470, 0.1358572244644165],
   [4133, 0.13212409615516663],
   [10033, 0.12530693411827087],
   [1016, 0.12288271635770798]]},
 'ja': {'layer_idx': 18,
  'top_layers_by_safety_score': [18, 25, 28, 7, 15],
  'safety_neurons_top_layer': [[10146, 0.38271063566207886],
   [10211, 0.19514533877372742],
   [10173, 0.19198766350746155],
   [6481, 0.18211539089679718],
   [5278, 0.17922574281692505],
   [1478, 0.15486685931682587],
   [9273, 0.1419661045074463],
   [4887, 0.13594770431518555],
   [1534, 0.1359312

## 5. The diagnostic: does `culture=us` suppress activation along the refusal direction?

And does each language's direction transfer to the *other* language's prompts? Four
combinations: direction-from-{en,ja} applied-to-{en,ja}.

In [ ]:

for direction_lang, (direction, layer_idx, _) in directions_per_lang.items():
    for test_lang in ["en", "ja"]:
        neutral_hs = last_token_hidden_states(data[test_lang]["harmful_neutral"])[layer_idx]
        us_hs = last_token_hidden_states(data[test_lang]["harmful_us"])[layer_idx]
        neutral_proj = neutral_hs.float() @ direction.float().cpu()
        us_proj = us_hs.float() @ direction.float().cpu()
        t_stat, p_val = stats.ttest_ind(neutral_proj.numpy(), us_proj.numpy())
        gap = float(neutral_proj.mean() - us_proj.mean())
        same_or_cross = "same-language" if direction_lang == test_lang else "CROSS-language transfer"
        print(f"direction from {direction_lang} (layer {layer_idx}) -> {test_lang} prompts [{same_or_cross}]: "
              f"neutral_mean={neutral_proj.mean():.3f} us_mean={us_proj.mean():.3f} gap={gap:.3f} "
              f"t={t_stat:.2f} p={p_val:.5f}")
        results.setdefault("projection_tests", []).append({
            "direction_from": direction_lang, "applied_to": test_lang, "layer_idx": int(layer_idx),
            "neutral_mean": float(neutral_proj.mean()), "us_mean": float(us_proj.mean()),
            "gap": gap, "t_stat": float(t_stat), "p_value": float(p_val),
        })


direction from en (layer 18) -> en prompts [same-language]: neutral_mean=4.995 us_mean=2.480 gap=2.516 t=1.77 p=0.09764

direction from en (layer 18) -> ja prompts [CROSS-language transfer]: neutral_mean=5.250 us_mean=4.949 gap=0.301 t=0.18 p=0.86154

direction from ja (layer 18) -> en prompts [CROSS-language transfer]: neutral_mean=-1.384 us_mean=-2.414 gap=1.030 t=0.72 p=0.48156

direction from ja (layer 18) -> ja prompts [same-language]: neutral_mean=1.341 us_mean=-2.781 gap=4.122 t=1.80 p=0.09270

## 6. Save results for pipeline step 4 (the fix targets these layers/neurons)

In [ ]:
with open("/content/pipeline2_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved /content/pipeline2_results.json")
print(json.dumps(results, indent=2))


Saved /content/pipeline2_results.json

{
  "en": {
    "layer_idx": 18,
    "top_layers_by_safety_score": [
      23,
      24,
      27,
      15,
      18
    ],
    "safety_neurons_top_layer": [
      [
        4198,
        0.32846102118492126
      ],
      [
        10045,
        0.21222341060638428
      ],
      [
        2297,
        0.18264585733413696
      ],
      [
        2517,
        0.16521579027175903
      ],
      [
        8225,
        0.15693753957748413
      ],
      [
        35,
        0.15670610964298248
      ],
      [
        3741,
        0.15396757423877716
      ],
      [
        265,
        0.15223166346549988
      ],
      [
        7830,
        0.14674922823905945
      ],
      [
        653,
        0.14152975380420685
      ],
      [
        7842,
        0.1413089483976364
      ],
      [
        1869,
        0.14031895995140076
      ],
      [
        8470,
        0.1358572244644165
      ],
      [
        4133,
        0.13212409615516663
      ],
      [
        1003